In [1]:
import time

import cv2
from matplotlib import pyplot as plt
import numpy as np

!python -m pip install -q ultralytics
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 3.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


## **1. Basic Tracking Run**

In [2]:
def process_clip(cap, model, fname, track=True):
    w,h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"(W,H): ({w},{h})")
    print(f"fps: {fps}")
    print(f"Total frames: {total}")

    start = time.time()
    frame_ctr = 0

    # Count the total unique IDs that appeared across all frames
    unique_ids = set()

    out = cv2.VideoWriter(fname,
                          cv2.VideoWriter_fourcc(*"mp4v"),
                          fps, (w, h))

    while cap.isOpened():
        ret, frame = cap.read() 
        if not ret: break

        if track:
            # Tracker used is ByteTrack (no ReID)
            results = model.track(frame, 
                                  persist=True, 
                                  classes=[2,3,5,7],
                                  conf=0.25,
                                  agnostic_nms=True,
                                  verbose=False,
                                  tracker="bytetrack.yaml")

        else:
            results = model.predict(frame,
                                    classes=[2,3,5,7],
                                    conf=0.25,  
                                    agnostic_nms=True, 
                                    verbose=False)
        
        annot_res = results[0].plot(font_size=17, pil=True)

        if results[0].boxes.id is not None:
            unique_ids.update(results[0].boxes.id.int().tolist())

        # VideoWriter.write expects frames in BGR
        out.write(cv2.cvtColor(np.array(annot_res), cv2.COLOR_RGB2BGR))
        frame_ctr += 1

        if frame_ctr % 200 == 0: 
            print(f"{frame_ctr}/{total} processed")

    elapsed = time.time() - start
    cap.release()
    out.release()

    print(f"Total unique IDs: {len(unique_ids)}")
    print(f"Duration: {elapsed:.1f}s")
    print(f"FPS: {frame_ctr/elapsed:.1f}")


def playback(cap):
    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        cv2.imshow("VideoPlayback", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

In [3]:
model = YOLO("yolov8m.pt")
cap1 = cv2.VideoCapture("../data/Dev_Clips/dev_overhead_medium_congest_01.mp4")
process_clip(cap1, model, "../outputs/output_track_overhead_medium_congest_01.mp4")

(W,H): (3840,2160)
fps: 24.000072000216
Total frames: 3000
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 240ms
Prepared 1 package in 31ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

200/3000 processed
400/3000 processed
600/3000 processed
800/3000 processed
1000/3000 processed
1200/3000 processed
1400/3000 processed
1600/3000 processed
1800/3000 processed
2000/3000 processed
2200/3000 processed
2400/3000 processed
2600/3000 processed
2800/3000 processed
Total unique IDs: 348
Duration: 929.9s
FPS: 3.1


### **Observations**

**Note:** The annotated output clip produced in this run is available [here](https://youtu.be/orn_LDkvYcQ). The frame numbers, scenarios, and other tracking/detection results referenced in the observations below are from this specific clip. Running the same code may produce slightly different results due to differences in hardware.

#### **General**
- The clip shows medium traffic (counted as per the project-defined vehicles/lane/minute thresholds)
- One outgoing lane is congested almost throughout the entirety of the 2-minute video
- Track IDs (via ByteTrack) stay consistent as vehicles move across frames (even as vehicles change lanes) in this moderate traffic clip
- However cases of tracking and detection instabilities can still be observed. Below are some examples.

#### **ID Transfer Case**
- Observed a case of ID transfer at frames 650-725 (0:00:26-:29) between two vehicles in the far zone. A car box had a detection dropout, and its id-552 was absorbed by another generated box close by.
- **What likely happened:** Vehicle 1 had a detection dropout for a few frames --> the tracker kept its predicted position via Kalman --> vehicle 2’s detection was closer to that predicted position and took it.
- Note that this ID transfer seen at frames 650-757 does not affect accurate counting since vehicle 1 was eventually assigned a new fresh ID as it got closer.

#### **Cascading ID Transfers**
- Seen at frames 1450-1650 when an outgoing truck passes next to the congested right-most lane consisting of stationary trucks. Adjust playback to 0.25x speed for better observation.
  1. As the outgoing truck came into view, it was immediately assigned an ID 1078 (normal behavior)
  2. It transferred that ID to the first truck (`truck1`) in the congested lane. `truck1` changed ID twice before acquiring ID 1078. The outgoing truck is then newly-assigned to ID 1083.  
     &nbsp;&nbsp;a.&nbsp;&nbsp; Likely caused by a brief detection dropout on the outgoing truck  
     &nbsp;&nbsp;b.&nbsp;&nbsp; Meanwhile, its track's Kalman prediction overlapped with `truck1` on the right, which took the ID.
     <br>
     <img src="assets/cascade-1475.png" width="800">
     <br>
  3. The outgoing truck passes `truck2` then abruptly switches ID from 1083 to 1100. `truck2`'s ID 219 persisted. `truck2`'s box did not overlap with the outgoing vehicle enough to cause a bad matching
  4. Upon passing `truck3` is when the cascading effect got worse. A merged detection between the passing truck and `truck3` on the right briefly occurred, with the merged box inheriting ID 1100
     <br>
     <img src="assets/cascade-1525.png" width="800">
     <br>
  5. The merged detection is eventually corrected but ID 1100 was transferred to `truck3` for brief milliseconds.  
     &nbsp;&nbsp;a.&nbsp;&nbsp; The moment the boxes separate, truck3 is stationary and sitting right where the merged box was.    
     &nbsp;&nbsp;b.&nbsp;&nbsp; For this reason, it got the better IoU match and took ID 1100.    
  6. The outgoing truck is now treated as a new object and assigned ID 1117.
  7. It kept moving and eventually re-entered the region where track 1100’s Kalman prediction would have landed, reacquiring ID 1100 back (initially, a merged detection with `truck4`).
     <br>
     <img src="assets/cascade-1550.png" width="800">
     <br>
  8. Meanwhile, `truck3` is now an inferior match to track 1100 so it was assigned to ID 1126.
  9. The outgoing truck also caused `truck4` in the congested lane to lose its original ID and acquire a new one (from ID 2 to 1153)    
      <br>
      <br>
- **Likely root cause:** Tightly packed vehices naturally produce heavily overlapping boxes, which makes IoU alone insufficient in reliably distinguishing which detection belongs to which track, so ***one bad match in the Hungarian assignment cascades through the rest***. Note that in the Hungarian algorithm, one wrong cost (caused by a low *1 - IoUxconfidence* between the wrong track-detection pair) can distort the entire global cost minimization. This algorithm does not match independently because every pairing affects every other pairing. We might look into using a different tracker that utilizes ReID to help maintain correct tracker-detection pairings even with congestion.

- **The “cascading” part is that each collision’s damage propagates forward.** The outgoing truck exits collision 1 (`truck1`) with a wrong ID (1073-->1083), which it brings into collision 2 (`truck2`), which produces another wrong ID (1100), which feeds into collision 3 then 4. If collision 1 produced correct results, collisions 2 and 3 either might never have happened or at least would be independent errors rather part of a chain.

#### **No ID Swaps are Observed**
- A true ID swap is when two vehicles cleanly exchange IDs with each other (vehicle A gets B’s ID, while vehicle B gets A’s ID at the same time).
- What we have observed so far was a series of ID transfers and reassignments that are one-directional (e.g., outgoing truck's ID is transfered to `truck1` before being assigned a new ID)
- While the IDs moved around many times, no two vehicles swapped IDs with each other in one single step.
- It is also possible I may have missed one or two instances while reviewing the two-minute video. But such occurrences are infrequent enough that they are not immediately observable in visual inspection.

#### **IDs Being Assigned Immediately on New Detections**
- Looking at the output vid, an ID seems to be assigned immediately as soon as a box appears for a vehicle, even if the initial box has a low confidence score (0.2-0.3)
- Might be worth looking at the ByteTrack config file to know precisely when IDs are assigned.
- Looking at the config also reveals the default value of other important parameters like the number of no-detection frames before a track is lost.

In [4]:
# View ByteTrack's config yaml file to see default parameters
import yaml
from pathlib import Path
import ultralytics

# Point to the cfg/trackers directory inside the ultralytics installation
cfg = yaml.safe_load(
    (Path(ultralytics.__file__).parent / "cfg" / "trackers" / "bytetrack.yaml").read_text())

for k, v in cfg.items():
    print(f"{k}: {v}")

tracker_type: bytetrack
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.25
track_buffer: 30
match_thresh: 0.8
fuse_score: True


### **Observations (Cont.)**

- `new_track_thresh: 0.25` means new, untracked boxes with conf scores >=0.25 is immediately assigned a track ID.
- This immediate ID assignment explains how low-confidence vehicles, as observed in the output clip, can get an ID right away (it was not a bug)
- The default `track_high_thresh: 0.25` is lower than the original [ByteTrack paper's](https://arxiv.org/abs/2110.06864) 0.6.
- This lower `track_high_threshold` can mean that almost every detection goes through ByteTrack's high-confidence first-pass Hungarian cost algorithm.
- A `track_buffer` of 30 means a track can maintain non-pairing with a detection for 30 frames. It stays alive for that many frames before it loses its track ID
- The clip is 25 fps, meaning, an established track can be lost in just above 1 second.

## **Compare With and Without Agnostic NMS**

So far, we have been observing results that was generated with `agnostic_nms=True`. This suppresses duplicate boxes of different classes on the same vehicle. As we have seen in the `detection-running-video-analysis` experiment notebook, this is a clear win for detection. However, NMS that suppresses a box also removes a detection the tracker might have needed in order to maintain a track during occlusion. For instance, say vehicle A is partially occluded behind vehicle B. The detector might produce two overlapping boxes -- one for A (low confidence) and one for B (high confidence). With `agnostic_nms=True`, A’s box can be suppressed because it highly overlaps with B’s higher-scoring box, even though they are two different vehicles. In addition, if NMS filters out a box that was the best IoU match for an existing track, that track goes unmatched for that frame and starts its lost-track countdown (`track_buffer`). This experiment, using the same clip as above, checks whether the detection-level fix of suppressing different-class boxes creates a tracking-level problem. 

In [3]:
# Modify process_clip function to make agnostic_nms parameter configurable
def process_clip(cap, model, fname, track=True, ag_nms=True):
    w,h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"(W,H): ({w},{h})")
    print(f"fps: {fps}")
    print(f"Total frames: {total}")

    start = time.time()
    frame_ctr = 0

    # Count the total unique IDs that appeared across all frames
    unique_ids = set()

    out = cv2.VideoWriter(fname,
                          cv2.VideoWriter_fourcc(*"mp4v"),
                          fps, (w, h))

    while cap.isOpened():
        ret, frame = cap.read() 
        if not ret: break

        if track:
            # Tracker used is ByteTrack (no ReID)
            results = model.track(frame, 
                                  persist=True, 
                                  classes=[2,3,5,7],
                                  conf=0.25,
                                  agnostic_nms=ag_nms,
                                  verbose=False,
                                  tracker="bytetrack.yaml")

        else:
            results = model.predict(frame,
                                    classes=[2,3,5,7],
                                    conf=0.25,  
                                    agnostic_nms=True, 
                                    verbose=False)
        
        annot_res = results[0].plot(font_size=17, pil=True)

        if results[0].boxes.id is not None:
            unique_ids.update(results[0].boxes.id.int().tolist())

        # VideoWriter.write expects frames in BGR
        out.write(cv2.cvtColor(np.array(annot_res), cv2.COLOR_RGB2BGR))
        frame_ctr += 1

        if frame_ctr % 200 == 0: 
            print(f"{frame_ctr}/{total} processed")

    elapsed = time.time() - start
    cap.release()
    out.release()

    print(f"Total unique IDs: {len(unique_ids)}")
    print(f"Duration: {elapsed:.1f}s")
    print(f"FPS: {frame_ctr/elapsed:.1f}")


def playback(cap):
    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        cv2.imshow("VideoPlayback", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

In [4]:
model.predictor = None
cap2 = cv2.VideoCapture("/kaggle/input/datasets/reubenjosephisidoro/dev-clips/Dev_Clips/dev_overhead_medium_congest_01.mp4")
process_clip(cap2, model, "output_track_overhead_medium_congest_01_nmsfalse.mp4", True, False)

(W,H): (3840,2160)
fps: 24.000072000216
Total frames: 3000
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 242ms
Prepared 1 package in 39ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

200/3000 processed
400/3000 processed
600/3000 processed
800/3000 processed
1000/3000 processed
1200/3000 processed
1400/3000 processed
1600/3000 processed
1800/3000 processed
2000/3000 processed
2200/3000 processed
2400/3000 processed
2600/3000 processed
2800/3000 processed
Total unique IDs: 470
Duration: 948.7s
FPS: 3.0


## **Observations**

**Note:** The annotated output clip produced in this run is available [here](https://youtu.be/vmwfw2UrEho). The frame numbers, scenarios, and other tracking/detection results referenced in the observations below are from this specific clip. Running the same code may produce slightly different results due to differences in hardware.

### **Difference in Number of Distinct Tracks Produced**
- The total unique IDs increased from 348 to 470 (a 35% increase) when `agnostic_nms` was turned off
- Per-class NMS suppressed duplicates within each class IoU-wise as intended. Still, it cannot touch cross-class duplicates, so a vehicle that saw a detection of two classes will keep both boxes.
- Each overlapping, duplicate box on the same vehicle spawns its own track and ID.
- Multi-box detections appeared mostly on vehicles where the model was undecided on class (e.g., two vans having a car and truck box at the same time seen at frames 1375-1425). Each box spawned its own track ID.  
  <br>
  <img src="assets/duplicate-detect-1375.png">
  <br>
### **Box Suppression During Occlusions**
- `agnostic_nms=False` had no meaningful effect in congested lanes because the congestion problem seems purely tracker-side (IoU matching + Hungarian algorithm) instead of a detection suppression issue.
- For instance, the congested lane behavior at frames 1450–1650 remain identical. We can see the same cascade pattern with ID transfers and merged detections occuring.
- During heavy occlusion (e.g., ~frame 1900-2100, a large truck blocks vehicles in the congested lane), occluded detections were suppressed identically in both runs (agnostic vs non-agnostic NMS) instead of being boxed.
- `agnostic_nms=True` is the correct setting for counting. Turning it off only adds duplicate tracks without fixing the IoU congestion cascade.
- This means the detector simply is not producing those boxes and NMS is not the one killing them.
- Therefore, the theoretical concern that agnostic_nms=True might suppress detections the tracker needs during occlusion was not confirmed.

**Conclusion.** `agnostic_nms=True` is the more ideal setting for counting. Turning it off unneccesarily adds duplicate tracks without fixing the congestion cascade or improving track maintenance during occlusion.

## **Track ID Stability in the Reliable Detection Zone**

Using the first output clip with `agnostic_nms=True`, we inspect vehicles in the reliable detection zone at 0.25x speed. The *reliable detection zone* is close to the camera for both incoming and outgoing lanes. The last 1-2 seconds before incoming vehicles exit the view at normal speed.


### **Incoming Lanes**
- Track IDs are mostly stable in the left incoming lanes. Vehicles mostly maintained their assigned IDs until exit
- There are cases of class flickering (vehicle oscillating between two classes) but no ID changes or switches. Expected since traffic on the incoming side is moderate without occlusions
- ID switches observed in the previous experiment on `detection-running-video-analysis` occurred exclusively in the far zone (not in the reliable zone)

### **Outgoing Lanes**
- Vehicles appearing into view frequently start with oversized bounding boxes which are corrected after a few frames.
- Note that this is not a rare case. It appears the detector needs at least a few frames to settle on the correct box size for the newly entering vehicles.
- For smaller vehicles (non-trucks), IDs mostly survive and maintained going through the reliable zone, even next to the congested lane.
- For trucks, we see the same ID issues as observed in the above experiments. Their size trigger detection merges and cascading ID transfers in the congested lane.

Note that even if a vehicle entered the reliable zone with a wrong ID (for example, it took the ID of an overlapping car in the far zone), that vehicle may hold that wrong ID consistently as it goes through the reliable zone. The tracking itself seem stable, even though it has the wrong ID assignment.